In [1]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

## Using the model to make predictions

In [42]:
import pandas as pd

# condition = "control"
# condition = "treatment"
# input_data = pd.read_csv(f"./all_{condition}_seekerhelper_pairs.csv")
# input_data = pd.read_csv("N94_all_seekerhelper_pairs.csv")
input_data = pd.read_csv("N94_all_seekerhelperalternative.csv")
print(len(input_data))
input_data.head()

3842


,id,seeker_post,response_post,alternative_response
0,14_a1df3c7155d0438b9c4084b57b66c6e6_0_0,NaN,good evening I understand you're feeling isola...,I hear you've returned from a vacation. How wa...
1,14_a1df3c7155d0438b9c4084b57b66c6e6_0_1,"Yeah, it's just... everyone was with their fam...",I hear you. It can be difficult growing apart ...,NaN
2,14_a1df3c7155d0438b9c4084b57b66c6e6_0_2,"Yeah, it's tough. You know, during the holiday...",Perhaps they're feeling similarly. Waiting for...,It sounds like you're feeling really left out ...
3,14_a1df3c7155d0438b9c4084b57b66c6e6_0_3,But why should I always have to be the bigger ...,"Being the bigger person can feel burdensome, c...",NaN
4,14_a1df3c7155d0438b9c4084b57b66c6e6_0_4,"Yeah, exactly. It's like, why should I keep pu...",How would you like for them to show you they c...,NaN


In [43]:
pd.isna(input_data.iloc[1].alternative_response)

True

In [44]:
from datasets import Dataset

study_dataset = Dataset.from_pandas(input_data)

In [45]:
from transformers import pipeline
import json

classifier_predict_config = {
    "Empathy-goodareas": {
        'model': "./roberta-Empathy-goodareas-eval_FeedbackESConv5pp_CARE10pp-sweeps-best-3lsljr3b-1741257767",
        'context_size': 1 # double check
    },
    "Reflections-goodareas": {
        # 'model': "./roberta-Reflections-goodareas-eval_FeedbackESConv5pp_CARE10pp-sweeps-best-d6x1jzik-1741277930",
        'model': "youralien/roberta-Reflections-goodareas-eval_FeedbackESConv5pp_CARE10pp-sweeps-current",
        'revision': 'f6238b84a06645bf280614b71bae0efb8a77afa3',
        'context_size': 1 # double check
    },
    "Questions-goodareas": {
        'model': "./roberta-Questions-goodareas-eval_FeedbackESConv5pp_CARE10pp-sweeps-best-82jc07j0-1741329550",
        'context_size': 1 # double check
    },
    "Validation-goodareas": {
        'model': "./roberta-Validation-goodareas-eval_FeedbackESConv5pp_CARE10pp-sweeps-best-wdbkc6pj-1741680290",
        'context_size': 3,
    },
    "Suggestions-badareas": {
        'model': "./roberta-Suggestions-badareas-eval_FeedbackESConv5pp_CARE10pp-sweeps-best-qpkjg3iw-1741352101",
        'context_size': 3
    },
    "Self-disclosure-badareas": {
        'model': "./roberta-Self-disclosure-badareas-eval_FeedbackESConv5pp_CARE10pp-sweeps-best-fk58yziy-1741688349",
        'context_size': 3 # interesting, roberta suffers from 512 token max context length, so had to artificially set this, lower than what it was at. 
    },
    "Suggestions-goodareas": {
        'model': "./roberta-Suggestions-goodareas-eval_FeedbackESConv5pp_CARE10pp-sweeps-best-6xtokquj-1741444633",
        'context_size': 3
    },
}

# Set which class to use
WHICH_CLASS = "Questions-goodareas"

# Load the appropriate model based on configuration
if 'revision' in classifier_predict_config[WHICH_CLASS]:
    classifier = pipeline("sentiment-analysis",model=classifier_predict_config[WHICH_CLASS]['model'],
                          revision=classifier_predict_config[WHICH_CLASS]['revision'],device=0)
else:
    classifier = pipeline("sentiment-analysis",model=classifier_predict_config[WHICH_CLASS]['model'],device=0)

def binary_prediction_seeker_response_post(seeker, helper):
    # sample = f"Seeker: {seeker}\nHelper: {helper}"
    sample = f"Seeker: {seeker}[SEP]Helper: {helper}"
    
    pred = classifier(sample)
    return int(pred[0]['label'] == 'selected')

Device set to use cuda:0


In [46]:
def predict_reflection(example):
    # Apply your binary prediction function to each example
    if pd.isna(example["alternative_response"]):
        example["prediction"] = 0  # no alternative response?  The category classifier would have not triggered
    else:
        example["prediction"] = binary_prediction_seeker_response_post(
            example["seeker_post"],
            example["alternative_response"] # NOTE: substituting the alternative response, to see what was done.
        )
    return example

# Apply the function to the entire dataset at once
predicted_dataset = study_dataset.map(predict_reflection)

Map: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3842/3842 [00:18<00:00, 205.12 examples/s]


In [47]:
input_data[f"{WHICH_CLASS}"] = predicted_dataset['prediction']

In [48]:
print(len(input_data))
input_data.head()

3842


,id,seeker_post,response_post,alternative_response,Questions-goodareas
0,14_a1df3c7155d0438b9c4084b57b66c6e6_0_0,NaN,good evening I understand you're feeling isola...,I hear you've returned from a vacation. How wa...,1
1,14_a1df3c7155d0438b9c4084b57b66c6e6_0_1,"Yeah, it's just... everyone was with their fam...",I hear you. It can be difficult growing apart ...,NaN,0
2,14_a1df3c7155d0438b9c4084b57b66c6e6_0_2,"Yeah, it's tough. You know, during the holiday...",Perhaps they're feeling similarly. Waiting for...,It sounds like you're feeling really left out ...,1
3,14_a1df3c7155d0438b9c4084b57b66c6e6_0_3,But why should I always have to be the bigger ...,"Being the bigger person can feel burdensome, c...",NaN,0
4,14_a1df3c7155d0438b9c4084b57b66c6e6_0_4,"Yeah, exactly. It's like, why should I keep pu...",How would you like for them to show you they c...,NaN,0


In [34]:
print(f"% Agreement between two different {WHICH_CLASS} classifers")
sum(input_data[f"{WHICH_CLASS}"] == input_data[f"{WHICH_CLASS}-1"]) / len(input_data)

% Agreement between two different Reflections-goodareas classifers


0.8456533055700156

In [49]:
print(f"N94_all_seekerhelperalternative_{WHICH_CLASS}.csv")
input_data.to_csv(f"N94_all_seekerhelperalternative_{WHICH_CLASS}.csv")

N94_all_seekerhelperalternative_Questions-goodareas.csv


In [37]:
f'all_{condition}_seekerhelper_pairs_{WHICH_CLASS}.csv'

'all_control_seekerhelper_pairs_Reflections-goodareas.csv'